# scikit-learn pipelines from raw table to model

A phone company wants to know which subscribers are about to leave (to **churn**). You will find the leak in a
colleague's script, write fit-on-train, transform-everywhere by hand, then build the real thing with
`ColumnTransformer`, `Pipeline` and `GridSearchCV`, save it, and fix next month's crash. Same invented
subscribers and same code as the lesson.

About 5 minutes. No account or key needed. Run the cells from top to bottom.

## 1. The subscribers

This cell makes the data. The company and its subscribers are invented: a small generator draws them with
seed 0. It writes two files next to the notebook: `subscribers_october.csv` (2000 subscribers, the table the
team trains on) and `subscribers_november.csv` (400 new subscribers for next month). November has a payment
method October never had.

**Look for:** 2000 October rows, churn rate 0.257, 156 blank `data_gb` values. November's payment values
include `'e-wallet'`. Subscriber T-10003 has `NaN` (a blank) in `data_gb`.

In [1]:
import numpy as np
import pandas as pd


def make(n, rng, start_id, payments, payment_p):
    tenure = rng.integers(1, 73, n)
    contract = rng.choice(["month-to-month", "one-year", "two-year"], n, p=[0.55, 0.25, 0.20])
    monthly_charge = np.round(rng.normal(65, 25, n).clip(15, 130), 2)
    support_calls = rng.poisson(1.6, n)
    data_gb = np.round(rng.gamma(2.0, 6.0, n), 1)
    payment = rng.choice(payments, n, p=payment_p)
    score = (-1.2 - 0.045 * tenure + 0.02 * (monthly_charge - 65) + 0.45 * support_calls
             + np.where(contract == "month-to-month", 1.3, np.where(contract == "one-year", 0.0, -1.0))
             + np.where(payment == "cash", 0.5, 0.0) + rng.normal(0, 0.8, n))
    churned = (rng.random(n) < 1 / (1 + np.exp(-score))).astype(int)
    df = pd.DataFrame({
        "customer_id": [f"T-{start_id + i}" for i in range(n)],
        "tenure_months": tenure, "monthly_charge": monthly_charge, "support_calls": support_calls,
        "data_gb": data_gb, "contract": contract, "payment": payment, "churned": churned,
    })
    # blanks: the usage meter misses some months
    df.loc[rng.random(n) < 0.08, "data_gb"] = np.nan
    return df


rng = np.random.default_rng(0)
october = make(2000, rng, 10000, ["card", "bank transfer", "cash"], [0.5, 0.35, 0.15])
november = make(400, rng, 20000, ["card", "bank transfer", "cash", "e-wallet"], [0.45, 0.3, 0.1, 0.15])
october.to_csv("subscribers_october.csv", index=False)
november.to_csv("subscribers_november.csv", index=False)
print("october rows", len(october), "churn rate", round(october.churned.mean(), 3), "blank data_gb", int(october.data_gb.isna().sum()))
print("november rows", len(november), "payment values", sorted(november.payment.unique()))
print(october.head(5).to_string())

october rows 2000 churn rate 0.257 blank data_gb 156
november rows 400 payment values ['bank transfer', 'card', 'cash', 'e-wallet']
  customer_id  tenure_months  monthly_charge  support_calls  data_gb        contract        payment  churned
0     T-10000             62           58.74              3      9.0  month-to-month           card        0
1     T-10001             46           55.48              2     17.2        two-year  bank transfer        0
2     T-10002             37           43.16              0      5.1  month-to-month           card        0
3     T-10003             20           35.17              1      NaN  month-to-month           cash        1
4     T-10004             23           66.49              2      7.2        one-year           card        0


## 2. The colleague's first script

This is the script from the lesson's first screen. It scales three columns, then splits off a quarter of the
rows as the test set.

**Look for:** `test accuracy: 0.782`. The bug is the line `StandardScaler().fit_transform(usage)`: the scaler
learns each column's mean and spread from all 2000 rows, test rows included, before the split.

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

subscribers = pd.read_csv("subscribers_october.csv")
usage = subscribers[["tenure_months", "monthly_charge",
                     "support_calls"]]
scaled = StandardScaler().fit_transform(usage)
X_train, X_test, y_train, y_test = train_test_split(
    scaled, subscribers["churned"], test_size=0.25, random_state=0)
model = LogisticRegression().fit(X_train, y_train)
print("test accuracy:", round(model.score(X_test, y_test), 4))

test accuracy: 0.782


## 3. Fixed: split first

The same script with the split moved first. The scaler is fitted on the training rows only, and the same
scaler then transforms the test rows.

**Look for:** `test accuracy: 0.782` again, exactly the same number. The next cell shows why.

In [3]:
subscribers = pd.read_csv("subscribers_october.csv")
usage = subscribers[["tenure_months", "monthly_charge",
                     "support_calls"]]
X_train, X_test, y_train, y_test = train_test_split(
    usage, subscribers["churned"], test_size=0.25, random_state=0)
scaler = StandardScaler().fit(X_train)
model = LogisticRegression().fit(scaler.transform(X_train), y_train)
X_test = scaler.transform(X_test)
print("test accuracy:", round(model.score(X_test, y_test), 4))

test accuracy: 0.782


## 4. Why the score did not move

We fit one scaler on all rows and one on the training rows only, and print what each learned. Then we train
both models and count the test predictions that differ.

**Look for:** the means and spreads differ only a little (for example `tenure_months`: 37.404 on all rows, 37.056
on the training rows). Of 500 test predictions, **0** differ. The leak was harmless here by luck, but the order
of the steps is still wrong.

In [4]:
TEST_SIZE = 0.25
usage = subscribers[["tenure_months", "monthly_charge", "support_calls"]]
X_train, X_test, y_train, y_test = train_test_split(usage, subscribers["churned"], test_size=TEST_SIZE, random_state=0)
all_rows = StandardScaler().fit(usage)
train_only = StandardScaler().fit(X_train)
for i, col in enumerate(usage.columns):
    print(f"{col}: mean all rows {all_rows.mean_[i]:.3f} | train rows {train_only.mean_[i]:.3f} ; "
          f"std all rows {all_rows.scale_[i]:.3f} | train rows {train_only.scale_[i]:.3f}")
leaky = LogisticRegression().fit(all_rows.transform(X_train), y_train).predict(all_rows.transform(X_test))
fixed = LogisticRegression().fit(train_only.transform(X_train), y_train).predict(train_only.transform(X_test))
print("test rows:", len(X_test), "| predictions that differ:", int((leaky != fixed).sum()))

tenure_months: mean all rows 37.404 | train rows 37.056 ; std all rows 20.616 | train rows 20.640
monthly_charge: mean all rows 66.127 | train rows 65.834 ; std all rows 24.456 | train rows 24.305
support_calls: mean all rows 1.607 | train rows 1.607 ; std all rows 1.256 | train rows 1.258
test rows: 500 | predictions that differ: 0


## 5. A pipeline from scratch: the steps

Now four number columns, `data_gb` included (it has blanks). Two steps learn numbers from data: `MeanFiller`
puts a column's mean into its blanks, and `Scaler` subtracts the mean and divides by the spread. Each has a
`fit` (learn the numbers) and a `transform` (apply them). `TinyLogistic` is a small logistic regression in
NumPy. The split is the same as before.

**Look for:** 1500 training rows and 500 test rows. The mean of `data_gb` is 12.0033 on the training rows,
12.1208 on all rows and 12.4685 on the test rows. 122 training blanks, 34 test blanks.

In [5]:
COLUMNS = ["tenure_months", "monthly_charge", "support_calls", "data_gb"]


class MeanFiller:
    name = "MeanFiller"
    def fit(self, rows):
        self.means = np.nanmean(rows, axis=0)
    def transform(self, rows):
        return np.where(np.isnan(rows), self.means, rows)


class Scaler:
    name = "Scaler"
    def fit(self, rows):
        self.means, self.stds = rows.mean(axis=0), rows.std(axis=0)
    def transform(self, rows):
        return (rows - self.means) / self.stds


class TinyLogistic:
    name = "TinyLogistic"
    def fit(self, rows, churn, lr=0.1, rounds=500):
        self.w, self.b = np.zeros(rows.shape[1]), 0.0
        for _ in range(rounds):
            p = 1 / (1 + np.exp(-(rows @ self.w + self.b)))
            self.w -= lr * rows.T @ (p - churn) / len(churn)
            self.b -= lr * np.mean(p - churn)
    def score(self, rows, churn):
        return float(np.mean(((rows @ self.w + self.b) > 0).astype(int) == churn))


rows = subscribers[COLUMNS].to_numpy(dtype=float)
train_rows, test_rows, train_churn, test_churn = train_test_split(
    rows, subscribers["churned"].to_numpy(), test_size=0.25, random_state=0)
print("train rows", len(train_rows), "test rows", len(test_rows))
print("mean data_gb: train rows", round(float(np.nanmean(train_rows[:, 3])), 4),
      "| all rows", round(float(np.nanmean(rows[:, 3])), 4),
      "| test rows", round(float(np.nanmean(test_rows[:, 3])), 4))
print("blank data_gb: train", int(np.isnan(train_rows[:, 3]).sum()), "test", int(np.isnan(test_rows[:, 3]).sum()))

train rows 1500 test rows 500
mean data_gb: train rows 12.0033 | all rows 12.1208 | test rows 12.4685
blank data_gb: train 122 test 34


## 6. Fit on training rows, apply everywhere

`fit_then_score` is the function you stepped through in the lesson. `fit` runs only on `train_rows`; the test
rows only go through `transform`. After it returns, we print what the steps learned and what the test column
looks like after each step.

**Look for:** the filler learned 12.0033 (the training mean). After filling, the test column averages 12.4368
with 0 blanks. After scaling, it averages 0.0544: close to 0 but not exactly, because it was scaled with the
training rows' numbers. Test accuracy 0.778.

In [6]:
def fit_then_score(steps, model, train_rows, train_churn, test_rows, test_churn):
    for step in steps:
        step.fit(train_rows)
        train_rows = step.transform(train_rows)
    model.fit(train_rows, train_churn)
    for step in steps:
        test_rows = step.transform(test_rows)
    return model.score(test_rows, test_churn)


steps = [MeanFiller(), Scaler()]
result = fit_then_score(steps, TinyLogistic(), train_rows, train_churn, test_rows, test_churn)
filler, scaler = steps
print("filler learned for data_gb (training mean):", round(float(filler.means[3]), 4))
filled = filler.transform(test_rows)
print("test data_gb after filling: blanks", int(np.isnan(filled[:, 3]).sum()), "mean", round(float(filled[:, 3].mean()), 4))
print("test data_gb after scaling: mean", round(float(scaler.transform(filled)[:, 3].mean()), 4))
print("test accuracy:", round(result, 4))

filler learned for data_gb (training mean): 12.0033
test data_gb after filling: blanks 0 mean 12.4368
test data_gb after scaling: mean 0.0544
test accuracy: 0.778


## 7. The real tool: ColumnTransformer, Pipeline, cross-validation

`ColumnTransformer` sends the number columns to a filler and a scaler, and the word columns to a one-hot
encoder. `customer_id` is not named, so it is left out. `Pipeline` joins that to the model. The scaler is a
`StandardScaler` that also counts how often it is fitted (only so we can see it).

**Look for:** 5 fold accuracies `[0.787 0.823 0.777 0.777 0.763]` and `scaler fits so far: 5`. Each fold
refitted the whole pipeline, scaler included, on its own training part.

In [7]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder


class CountingScaler(StandardScaler):
    fits = 0
    def fit(self, X, y=None, sample_weight=None):
        CountingScaler.fits += 1
        return super().fit(X, y, sample_weight)


subscribers = pd.read_csv("subscribers_october.csv")
X_train, X_test, y_train, y_test = train_test_split(
    subscribers.drop(columns="churned"), subscribers["churned"], test_size=0.25, random_state=0)

numbers = ["tenure_months", "monthly_charge", "support_calls", "data_gb"]
categories = ["contract", "payment"]
prepare = ColumnTransformer([
    ("numbers", make_pipeline(SimpleImputer(strategy="mean"),
                              CountingScaler()), numbers),
    ("categories", OneHotEncoder(), categories),
])
churn_model = Pipeline([("prepare", prepare), ("model", LogisticRegression())])
scores = cross_val_score(churn_model, X_train, y_train, cv=5)
print("cross-validation accuracy:", scores.round(3))
print("scaler fits so far:", CountingScaler.fits)

cross-validation accuracy: [0.787 0.823 0.777 0.777 0.763]
scaler fits so far: 5


## 8. Tune it, test it once, save it

`GridSearchCV` tries 4 values of `C` with 5-fold cross-validation. The key `"model__C"` means: step `model`,
parameter `C`. After choosing, it fits the pipeline once more on all training rows. `joblib.dump` saves that
fitted pipeline, steps and model together.

**Look for:** `best: {'model__C': 0.1} cv accuracy: 0.7853`, `scaler fits in the search: 21` (4 x 5 + 1) and
`test accuracy: 0.79`. The cross-validation score and the test score agree.

In [8]:
CountingScaler.fits = 0
search = GridSearchCV(churn_model, {"model__C": [0.01, 0.1, 1, 10]}, cv=5)
search.fit(X_train, y_train)
print("best:", search.best_params_, "cv accuracy:", round(search.best_score_, 4))
print("scaler fits in the search:", CountingScaler.fits)
print("test accuracy:", round(search.score(X_test, y_test), 4))
joblib.dump(search.best_estimator_, "churn_model.joblib")

best: {'model__C': 0.1} cv accuracy: 0.7853
scaler fits in the search: 21
test accuracy: 0.79


['churn_model.joblib']

## 9. What the model sees

`get_feature_names_out()` lists the columns that come out of the ColumnTransformer. We also print the mean
cross-validation accuracy for each `C`.

**Look for:** 10 columns: 4 starting with `numbers__` and 6 one-hot columns starting with `categories__`
(3 contract values, 3 payment values: bank transfer, card, cash). No `e-wallet` column, because October had
no e-wallet payers.

In [9]:
names = list(search.best_estimator_.named_steps["prepare"].get_feature_names_out())
print("columns the model sees:", len(names))
for name in names:
    print("  ", name)
print("mean cv accuracy per C:", dict(zip([0.01, 0.1, 1, 10], search.cv_results_["mean_test_score"].round(4).tolist())))

columns the model sees: 10
   numbers__tenure_months
   numbers__monthly_charge
   numbers__support_calls
   numbers__data_gb
   categories__contract_month-to-month
   categories__contract_one-year
   categories__contract_two-year
   categories__payment_bank transfer
   categories__payment_card
   categories__payment_cash
mean cv accuracy per C: {0.01: 0.76, 0.1: 0.7853, 1: 0.7853, 10: 0.784}


## 10. Next month: the rerun breaks (on purpose)

The colleague loads `churn_model.joblib` and scores November's 400 subscribers. This cell is meant to fail:
the `try` / `except` catches the error and prints its message, so the notebook keeps going.

**Look for:** 64 November subscribers pay by e-wallet, then
`ValueError: Found unknown categories ['e-wallet'] in column 1 during transform`. Column 1 is counted inside the
encoder, which only gets `contract` (0) and `payment` (1).

In [10]:
churn_model = joblib.load("churn_model.joblib")
november = pd.read_csv("subscribers_november.csv")
print("november payment values:", november["payment"].value_counts().to_dict())
try:
    risk = churn_model.predict_proba(november.drop(columns="churned"))[:, 1]
    print("subscribers scored:", len(risk))
except ValueError as error:
    print("ValueError:", error)

november payment values: {'card': 178, 'bank transfer': 115, 'e-wallet': 64, 'cash': 43}
ValueError: Found unknown categories ['e-wallet'] in column 1 during transform


## 11. The fix: tell the encoder how to handle unknowns

We rebuild the same pipeline with `OneHotEncoder(handle_unknown="ignore")`, refit the search on October, save
it again, load it and score November. An unseen value gets all zeros in the payment columns.

**Look for:** the search gives the same `best` and `test accuracy: 0.79` as before (October has no unknowns).
Then `subscribers scored: 400`, `flagged (risk over 0.5): 62` and `accuracy on november: 0.78`.

In [11]:
prepare = ColumnTransformer([
    ("numbers", make_pipeline(SimpleImputer(strategy="mean"),
                              CountingScaler()), numbers),
    ("categories", OneHotEncoder(handle_unknown="ignore"), categories),
])
churn_model = Pipeline([("prepare", prepare), ("model", LogisticRegression())])
search = GridSearchCV(churn_model, {"model__C": [0.01, 0.1, 1, 10]}, cv=5)
search.fit(X_train, y_train)
print("best:", search.best_params_, "cv accuracy:", round(search.best_score_, 4))
print("test accuracy:", round(search.score(X_test, y_test), 4))
joblib.dump(search.best_estimator_, "churn_model.joblib")

churn_model = joblib.load("churn_model.joblib")
risk = churn_model.predict_proba(november.drop(columns="churned"))[:, 1]
print("subscribers scored:", len(risk))
print("flagged (risk over 0.5):", int((risk > 0.5).sum()))
print("accuracy on november:", round(churn_model.score(november.drop(columns="churned"), november["churned"]), 4))

best:

 {'model__C': 0.1} cv accuracy: 0.7853
test accuracy: 0.79


subscribers scored: 400
flagged (risk over 0.5): 62
accuracy on november: 0.78


## Now change it

Each cell below is ready to run. Edit the one line marked `# <- change this`, run the cell, and compare.

### Experiment 1: fewer folds, fewer fits

Change `cv=5` to `cv=3`. The search should now fit the scaler 4 x 3 + 1 = **13** times instead of 21: one fit
per value of `C` per fold, plus the final refit on all training rows.

In [12]:
CountingScaler.fits = 0
search = GridSearchCV(churn_model, {"model__C": [0.01, 0.1, 1, 10]},
                      cv=5)  # <- change this: cv=3
search.fit(X_train, y_train)
print("scaler fits in the search:", CountingScaler.fits)
print("best:", search.best_params_, "cv accuracy:", round(search.best_score_, 4))

scaler fits in the search: 21
best: {'model__C': 0.1} cv accuracy: 0.7853


### Experiment 2: a tiny training set makes the leak bigger

This repeats cell 4 with a different test size. Change `0.25` to `0.98`, so only 40 rows are left for training.
The scaler fitted on all 2000 rows and the one fitted on 40 rows now learn clearly different means and
spreads, and the number of test predictions that differ should rise above 0. A leak is small when the test
rows are a small share of what the step learns from.

In [13]:
TEST_SIZE = 0.25  # <- change this: 0.98
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(usage, subscribers["churned"], test_size=TEST_SIZE, random_state=0)
all_rows = StandardScaler().fit(usage)
train_only = StandardScaler().fit(X_train_s)
for i, col in enumerate(usage.columns):
    print(f"{col}: mean all rows {all_rows.mean_[i]:.3f} | train rows {train_only.mean_[i]:.3f}")
leaky = LogisticRegression().fit(all_rows.transform(X_train_s), y_train_s).predict(all_rows.transform(X_test_s))
fixed = LogisticRegression().fit(train_only.transform(X_train_s), y_train_s).predict(train_only.transform(X_test_s))
print("train rows:", len(X_train_s), "| test rows:", len(X_test_s), "| predictions that differ:", int((leaky != fixed).sum()))

tenure_months: mean all rows 37.404 | train rows 37.056
monthly_charge: mean all rows 66.127 | train rows 65.834
support_calls: mean all rows 1.607 | train rows 1.607
train rows: 1500 | test rows: 500 | predictions that differ: 0


### Experiment 3: fill with the median instead of the mean

Change `strategy="mean"` to `strategy="median"`. The fill value for `data_gb`, learned from the training rows,
should drop from about 12.0 to about 10: the column has a long tail of heavy users, which pulls the mean up but
not the median. The cross-validation accuracy should barely move, because only 8% of the values are blank.

In [14]:
filler_choice = SimpleImputer(strategy="mean")  # <- change this: strategy="median"
prepare = ColumnTransformer([
    ("numbers", make_pipeline(filler_choice, StandardScaler()), numbers),
    ("categories", OneHotEncoder(handle_unknown="ignore"), categories),
])
model_3 = Pipeline([("prepare", prepare), ("model", LogisticRegression(C=0.1))])
print("cross-validation accuracy:", round(cross_val_score(model_3, X_train, y_train, cv=5).mean(), 4))
model_3.fit(X_train, y_train)
fill = model_3.named_steps["prepare"].named_transformers_["numbers"][0].statistics_
print("fill value learned for data_gb:", round(float(fill[3]), 4))

cross-validation accuracy: 0.7853
fill value learned for data_gb: 12.0033


## What you should see

- Cell 7: `cross_val_score` prints 5 accuracies between 0.763 and 0.823.
- Cells 7 and 8: the scaler counter prints 5 after cross-validation and 21 after the grid search.
- Cell 8: the search picks `{'model__C': 0.1}` with cross-validation accuracy 0.7853, and the test rows give 0.79.
- Cell 9: `get_feature_names_out()` lists 10 columns, each starting with `numbers__` or `categories__`.
- Cells 10 and 11: scoring November with the default encoder raises `ValueError` naming 'e-wallet'; with
  `handle_unknown="ignore"` all 400 subscribers are scored.